# 04 — Weather Modeling

**Phase 2, Step 3** of the forecasting pipeline.

**Input:**  `data/processed/weather_features.csv` (from notebook 03)
**Output:** `models/weather/<target>/<algo>.joblib`, `summary.csv`, `best.json`

---

## Ladder of models

We evaluate models of increasing power and keep the best one per target:

| Algo | Purpose |
|---|---|
| **Persistence** | `y[t+h] = y[t]` — the floor every model must beat |
| **Ridge** | Regularised linear — interpretable baseline |
| **HGBR** | `HistGradientBoostingRegressor` — fast GBM, handles NaNs natively |
| **XGBoost** | Used if `xgboost` is installed; same class of model, often slightly better |
| **SARIMA / Prophet** | Per-city classical time-series baselines (optional) |

## Targets

We forecast 5 daily quantities:

| Target | Daily column | Special handling |
|---|---|---|
| `temperature_2m` | `temperature_2m_mean` | — |
| `wind_speed_10m` | `wind_speed_10m_mean` | — |
| `wind_direction_10m` | `wind_direction_10m` | **circular MAE** (0-360 wrap) |
| `rain` | `rain_sum` | Heavy-tailed |
| `precipitation` | `precipitation_sum` | Heavy-tailed |

## How leakage is prevented

Every feature was built with `.shift(1).rolling(...)` per city (notebook 03).
On top of that, this notebook shifts the target **forward** by the forecast
horizon before training:

```
X at row t  (features observed through day t)
y at row t  =  target[t + horizon]    # what we want to predict
```

The train/test split is strictly by feature-row date — no future information
bleeds into training.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

from src.weather import train as tr
from src.weather.features import DAILY_TARGET_COLUMNS
from src.utils.config import PROCESSED_DIR, MODELS_DIR, FORECAST_TARGETS
from src.utils.logging_utils import get_logger
logger = get_logger("nb.04_modeling")

## 2. Load features and show the training schema

In [ ]:
feats = pd.read_csv(PROCESSED_DIR / "weather_features.csv", parse_dates=["date"])
if feats["date"].dt.tz is not None:
    feats["date"] = feats["date"].dt.tz_localize(None)
print(f"Feature matrix: {feats.shape}")
print(f"Forecast targets: {FORECAST_TARGETS}")

In [ ]:
# Peek at how `prepare_training_frames` shifts targets and splits
tf = tr.prepare_training_frames(
    feats, target="temperature_2m", test_start="2025-01-01", horizon=1,
)
print(f"\nTarget: {tf.target_col} (h={tf.horizon})")
print(f"X_train: {tf.X_train.shape}  y_train: {tf.y_train.shape}")
print(f"X_test : {tf.X_test.shape}   y_test : {tf.y_test.shape}")
print(f"\nFeature-row train window: {tf.train_meta['date'].min().date()}  ->  {tf.train_meta['date'].max().date()}")
print(f"Feature-row test  window: {tf.test_meta['date'].min().date()}  ->  {tf.test_meta['date'].max().date()}")

## 3. Train individual models for temperature (walkthrough)

Every trainer returns `(fitted_model, test_predictions)`.  We'll evaluate each
against the same test set and compare.

In [ ]:
results_temp = {}
for name, trainer in [
    ("persistence", tr.train_persistence),
    ("ridge",       tr.train_ridge),
    ("hgbr",        tr.train_hgbr),
]:
    model, pred = trainer(tf)
    m = tr.evaluate_regression(tf.y_test.values, pred)
    results_temp[name] = {"model": model, "pred": pred, "metrics": m}
    print(f"  {name:12s} MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  R2={m['R2']:.4f}")

### 3a. Diagnostic: predicted vs actual for the best model

In [ ]:
best_name = min(results_temp, key=lambda k: results_temp[k]["metrics"]["RMSE"])
pred = results_temp[best_name]["pred"]
y = tf.y_test.values

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].scatter(y, pred, s=3, alpha=0.25, c="steelblue")
lo, hi = min(y.min(), pred.min()), max(y.max(), pred.max())
ax[0].plot([lo, hi], [lo, hi], "k-", lw=1, alpha=0.6, label="perfect")
ax[0].set_xlabel("actual (°C)"); ax[0].set_ylabel("predicted (°C)")
ax[0].set_title(f"temperature_2m  —  {best_name} (MAE={results_temp[best_name]['metrics']['MAE']:.2f}°C)")
ax[0].legend(); ax[0].grid(alpha=0.3)

# Residuals vs date
res = y - pred
ax[1].scatter(tf.test_meta["date"], res, s=3, alpha=0.3, c="darkred")
ax[1].axhline(0, c="k", lw=1)
ax[1].set_xlabel("date"); ax[1].set_ylabel("residual (°C)")
ax[1].set_title("Residuals vs time — look for systematic bias")
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Full orchestrator — all targets × all algos

`train_weather_models` iterates over every (target, algo) combination,
picks the best by test RMSE, and writes artefacts to `models/weather/`.

In [ ]:
summary = tr.train_weather_models(
    algos=["persistence", "ridge", "hgbr", "xgboost"],
    run_cv=True,
    horizon=1,
)
summary.round(3).style.background_gradient(cmap="RdYlGn_r", subset=["RMSE"]) \
       .format(precision=3)

### 4a. Best algo per target (from `models/weather/best.json`)

In [ ]:
import json as _json
best = _json.loads((MODELS_DIR / "weather" / "best.json").read_text())
pd.DataFrame([
    {"target": t, "best_algo": d["algo"],
     "RMSE": d["metrics"].get("RMSE"),
     "MAE":  d["metrics"].get("MAE"),
     "R2":   d["metrics"].get("R2"),
     "model_path": d["model_path"]}
    for t, d in best.items()
])

## 5. Feature importance — what drives the temperature model?

In [ ]:
import joblib

model_path = MODELS_DIR / "weather" / "temperature_2m" / "hgbr.joblib"
if model_path.exists():
    model = joblib.load(model_path)
    # sklearn's HistGradientBoostingRegressor doesn't expose feature_importances_,
    # but permutation importance on a small validation slice is always available.
    from sklearn.inspection import permutation_importance
    tf_t = tr.prepare_training_frames(feats, "temperature_2m", test_start="2025-01-01", horizon=1)
    # Use a subsample to keep permutation fast
    rng = np.random.default_rng(0)
    idx = rng.choice(len(tf_t.X_test), size=min(500, len(tf_t.X_test)), replace=False)
    imp = permutation_importance(
        model, tf_t.X_test.iloc[idx], tf_t.y_test.iloc[idx],
        n_repeats=5, random_state=0, n_jobs=1,
    )
    imp_df = pd.DataFrame({
        "feature": tf_t.X_test.columns,
        "importance": imp.importances_mean,
        "std": imp.importances_std,
    }).sort_values("importance", ascending=False).head(20)
    print(imp_df.to_string(index=False))
else:
    print(f"Model not found at {model_path} — run the orchestrator above first.")

## 6. Circular metric sanity check — wind direction

For `wind_direction_10m` a model predicting 10° vs a truth of 350° is only
20° wrong (wrapping around north), not 340°. The orchestrator uses
`circular=True` automatically for this target.

In [ ]:
# Run a quick side-by-side to confirm
from src.weather.train import _circular_abs_error_deg
y_t = np.array([350.0, 10.0, 179.0])
y_p = np.array([ 10.0, 350.0, 180.0])
print("Scalar abs error   :", np.abs(y_t - y_p))
print("Circular abs error :", _circular_abs_error_deg(y_t, y_p))

## 7. Multi-horizon extension (optional)

Our training harness takes a `horizon` argument. The orchestrator currently
trains only at h=1 (tomorrow). To train a full 1-30 day ladder:

```python
horizon_summaries = []
for h in (1, 3, 7, 14, 30):
    s = tr.train_weather_models(horizon=h, run_cv=False,
                                algos=["persistence", "ridge", "hgbr"])
    horizon_summaries.append(s)
```

We'll wire this into the forecasting step (notebook 05) where we need to
predict across a full 30-day horizon anyway.

## 8. Persisted artefacts

In [ ]:
from pathlib import Path
artefacts = sorted((MODELS_DIR / "weather").rglob("*"))
rows = []
for p in artefacts:
    if p.is_file():
        rows.append({"path": str(p.relative_to(MODELS_DIR)), "size_KB": round(p.stat().st_size / 1024, 1)})
pd.DataFrame(rows)

---
**Step 3 of Phase 2 ✅ complete.** Next: `05_weather_forecasting.ipynb` — produce a
30-day forward forecast by recursively applying the trained models, and persist
`data/processed/weather_forecast.csv`.